# FER2013 — Kaggle Notebook runner

Runs entirely on Kaggle — the competition data is mounted, so there's no `kaggle.json` / download step.

**Before running, in the right-hand panel:**
1. **Settings → Accelerator → GPU** (P100 or T4 x2).
2. **Settings → Internet → On** (needed for wandb, `git clone`, and pretrained weights; requires a phone-verified Kaggle account).
3. **Add Input** → search `challenges-in-representation-learning-facial-expression-recognition-challenge` → Add.
4. **Add-ons → Secrets** → add a secret named `WANDB_API_KEY` (from https://wandb.ai/authorize).

Then edit `REPO_URL` below. To run overnight in the background: **Save Version → Save & Run All (Commit)**.

In [ ]:
!nvidia-smi

## 1. Clone your GitHub repo (edit REPO_URL)

In [ ]:
REPO_URL = 'https://github.com/<you>/<repo>.git'  # <-- EDIT THIS
import os
repo = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
%cd /kaggle/working
if not os.path.exists(repo):
    !git clone {REPO_URL}
%cd {repo}
!ls

## 2. Install wandb

In [ ]:
!pip install -q wandb pyyaml

## 3. Wandb login (from the Kaggle Secret) — no interactive prompt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'fer2013'
print('wandb key loaded:', bool(os.environ.get('WANDB_API_KEY')))

## 4. Locate the mounted competition data

In [ ]:
import glob, os
cands = glob.glob('/kaggle/input/*expression*') + glob.glob('/kaggle/input/*facial*')
base = cands[0] if cands else '/kaggle/input'

def _find(name):
    hits = glob.glob(f'{base}/**/{name}', recursive=True)
    return hits[0] if hits else None

TRAIN_CSV = _find('train.csv')
TEST_CSV = _find('test.csv')

if TRAIN_CSV is None:  # fallback: build splits from fer2013.csv (has a Usage column)
    import pandas as pd
    fer = _find('fer2013.csv')
    df = pd.read_csv(fer)
    os.makedirs('/kaggle/working/data', exist_ok=True)
    df[df.Usage == 'Training'][['emotion', 'pixels']].to_csv('/kaggle/working/data/train.csv', index=False)
    df[df.Usage != 'Training'][['pixels']].to_csv('/kaggle/working/data/test.csv', index=False)
    TRAIN_CSV, TEST_CSV = '/kaggle/working/data/train.csv', '/kaggle/working/data/test.csv'

print('train:', TRAIN_CSV)
print('test :', TEST_CSV)

## 5. Sanity checks (forward / backward) + a short run

In [ ]:
!python -m src.train --config configs/02_tiny_cnn.yaml --train-csv {TRAIN_CSV} --run-sanity --epochs 5 --tag sanity --wandb-project fer2013

## 6. Run all experiments
Each becomes its own grouped Wandb run. Set `QUICK_EPOCHS = 5` for a smoke test first.

In [ ]:
EXPERIMENTS = [
    '01_mlp',
    '02_tiny_cnn',
    '03_vgg_plain',
    '04_vgg_reg',
    '05_resnet_mini',
    '06_resnet18_transfer',
]
QUICK_EPOCHS = None  # e.g. 5 for a smoke test; None = use each config's epochs

for name in EXPERIMENTS:
    print('\n' + '=' * 70 + f'\nRUNNING {name}\n' + '=' * 70)
    extra = f'--epochs {QUICK_EPOCHS}' if QUICK_EPOCHS else ''
    !python -m src.train --config configs/{name}.yaml --train-csv {TRAIN_CSV} --test-csv {TEST_CSV} --wandb-project fer2013 {extra}

## 7. (Optional) Hyperparameter sweep — run this interactively, NOT in commit mode
```
!wandb sweep configs/sweep_vgg.yaml
!wandb agent <entity>/fer2013/<sweep_id> --count 10
```

## 8. Build a Kaggle submission

In [ ]:
!python -m src.predict --checkpoint outputs/04_vgg_reg/best.pth --test-csv {TEST_CSV} --out /kaggle/working/submission.csv
print('submission saved to /kaggle/working/submission.csv')